# SECOM 반도체 불량 선별 · 9월 비교 파일럿

2008년 7~8월 센서 자료로 두 개의 RBF-SVM을 학습하고, 9월 590건에서 위험도 상위 10%를 사후에 선별한다. 전체 센서 RBF-SVM, 4개 주성분 RBF-SVM, 같은 4개 성분을 사용하는 양자 커널 SVM을 비교한다. 10월 센서 행과 정답은 모델 학습·선택·평가에서 제외한다.

**평가 규칙** · 위험 순위 상위 59건에서 포착한 불량 건수, 불량 재현율, 전체 순위의 평균정밀도(AP). 같은 점수는 원자료 행 순서로 정렬한다. 재현율은 적발된 불량 수를 그달의 전체 불량 수로 나눈다. AP는 불량의 순위 전반을 요약한다. 고정 모델에 대한 불확실성은 날짜 단위로 두 모델의 결과를 함께 재표집해 추정한다.

**실행** · Python, NumPy, SciPy, scikit-learn, threadpoolctl을 설치한 Jupyter 또는 Colab에서 위에서 아래로 실행한다. 같은 폴더나 `data/`에 두 원본 파일을 놓거나, 같은 폴더에 `secom.zip`을 놓을 수 있다. 파일이 없을 때는 UCI의 공개 ZIP을 가져온다.

[McCann & Johnston, UCI SECOM, DOI 10.24432/C54305](https://archive.ics.uci.edu/dataset/179/secom) · [가장 가까운 QSVM 선행연구](https://doi.org/10.1371/journal.pone.0361400)

In [1]:
from datetime import datetime
from itertools import product
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
import hashlib
import math
import platform
import numpy as np
import scipy
import sklearn
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits

names = ("secom.data", "secom_labels.data")
def read_source():
    for root in (Path("."), Path("data"), Path("../data")):
        if all((root / name).is_file() for name in names):
            return [(root / name).read_bytes() for name in names]
    if Path("secom.zip").is_file():
        archive_data = Path("secom.zip").read_bytes()
    else:
        url = "https://archive.ics.uci.edu/static/public/179/secom.zip"
        with urlopen(url, timeout=45) as response:
            archive_data = response.read()
    with ZipFile(BytesIO(archive_data)) as archive:
        by_name = {Path(entry).name: entry for entry in archive.namelist()}
        return [archive.read(by_name[name]) for name in names]

raw_x, raw_labels = read_source()
print("Python", platform.python_version(), "| NumPy", np.__version__, "| SciPy", scipy.__version__, "| scikit-learn", sklearn.__version__)
for name, raw in zip(names, (raw_x, raw_labels)):
    print(name, "SHA-256", hashlib.sha256(raw).hexdigest())
x_lines, label_lines = raw_x.splitlines(), raw_labels.splitlines()
assert len(x_lines) == len(label_lines) == 1567
selected_x, selected_y, selected_dates = [], [], []
for row_id, label_line in enumerate(label_lines):
    raw_label, raw_date = label_line.decode("ascii").split(maxsplit=1)
    stamp = datetime.strptime(raw_date.strip('"'), "%d/%m/%Y %H:%M:%S").date()
    if stamp.month in (7, 8, 9):
        assert raw_label in ("1", "-1")
        selected_x.append(x_lines[row_id])
        selected_y.append(int(raw_label == "1"))
        selected_dates.append(stamp)
X = np.loadtxt(BytesIO(b"\n".join(selected_x)))
y = np.asarray(selected_y)
dates = np.array(selected_dates, dtype="datetime64[D]")
train = dates < np.datetime64("2008-09-01")
september = ~train
assert X.shape == (1208, 590)
assert (train.sum(), y[train].sum(), september.sum(), y[september].sum()) == (618, 65, 590, 17)
print("Jul-Aug train:", train.sum(), "failures:", y[train].sum())
print("Sep validation:", september.sum(), "failures:", y[september].sum())

Python 3.14.0 | NumPy 2.4.3 | SciPy 1.16.3 | scikit-learn 1.8.0
secom.data SHA-256 4503bdb42e63687d1c7586bdc1796df9a7cb5ae9cc9c0443d84957a446de3aaf
secom_labels.data SHA-256 2fb2e8b5dfe77ba96ecf0960ff00f2ba9311795383a8342d4588a7cb37c816ef
Jul-Aug train: 618 failures: 65
Sep validation: 590 failures: 17


## 7~8월 자료 안에서 설정 선택

8월 8~15일, 16~23일, 24~31일을 차례로 내부 검증 기간으로 쓰고 각 구간의 시작일 이전 자료로 학습한다. 매번 학습 자료로 중앙값 대치, 상수 열 제거, 표준화, 선택적으로 PCA를 적합한다. 결측치뿐인 학습 열은 대치 단계에서 제외한다. PCA는 비백색화 방식으로 적합한다.

두 모델 모두 클래스 가중치를 balanced로 설정하고, C 후보 3개와 RBF 폭 후보 3개를 탐색한다. 감마는 해당 모델 학습 입력에 대한 기본값 1/(차원 수 × 분산)에 배수를 곱한다. 세 기간별 AP의 단순평균이 가장 높은 조합을 고른다. 동률이면 먼저 나열된 작은 C와 배수를 택한다.

In [2]:
from warnings import catch_warnings, filterwarnings

fold_dates = [("2008-08-08", "2008-08-16"), ("2008-08-16", "2008-08-24"), ("2008-08-24", "2008-09-01")]
candidates = list(product((0.1, 1.0, 10.0), (0.25, 1.0, 4.0)))
modes = ("All sensors", "PCA-4")

def transform(fit_rows, eval_rows, mode):
    steps = [SimpleImputer(strategy="median"), VarianceThreshold(), StandardScaler()]
    if mode == "PCA-4":
        steps.append(PCA(n_components=4, svd_solver="full"))
    transformer = make_pipeline(*steps)
    with catch_warnings():
        filterwarnings("ignore", message="Skipping features without any observed values.*", category=UserWarning)
        fit_values = transformer.fit_transform(X[fit_rows])
        eval_values = transformer.transform(X[eval_rows])
    return fit_values, eval_values, transformer

def svm_scores(fit_values, fit_labels, eval_values, params):
    c, multiplier = params
    gamma = multiplier / (fit_values.shape[1] * fit_values.var())
    model = SVC(C=c, gamma=gamma, kernel="rbf", class_weight="balanced")
    model.fit(fit_values, fit_labels)
    return model.decision_function(eval_values)

chosen = {}
with threadpool_limits(limits=1):
    for mode in modes:
        fold_ap = {p: [] for p in candidates}
        for start, end in fold_dates:
            fit = dates < np.datetime64(start)
            val = (dates >= np.datetime64(start)) & (dates < np.datetime64(end))
            assert set(np.unique(y[val])) == {0, 1}
            fit_x, val_x, _ = transform(fit, val, mode)
            for params in candidates:
                score = svm_scores(fit_x, y[fit], val_x, params)
                fold_ap[params].append(average_precision_score(y[val], score))
        chosen[mode] = max(candidates, key=lambda p: np.mean(fold_ap[p]))
        print(mode, "C / gamma multiplier:", chosen[mode], "| fold AP:", np.round(fold_ap[chosen[mode]], 4))

All sensors C / gamma multiplier: (1.0, 0.25) | fold AP: [0.545  0.1066 0.0973]
PCA-4 C / gamma multiplier: (1.0, 1.0) | fold AP: [0.3844 0.1568 0.0583]


## 9월 고정 평가

훈련 구간 전체로 다시 학습한 뒤 9월의 순위를 한 번 산출한다. 검사량은 ceil(0.10 × 해당 기간 건수)로 정한다. 이는 월 단위 일괄 선별을 모사한다. 모든 날짜를 아우르는 평균을 실시간 경보 성능으로 해석하려면 별도 설계가 필요하다.

In [3]:
def metrics(labels, scores):
    n_check = math.ceil(0.10 * len(labels))
    top = np.argsort(-scores, kind="stable")[:n_check]
    hits = int(labels[top].sum())
    return top, hits, hits / labels.sum(), average_precision_score(labels, scores)

labels = y[september]
scores, top_sets = {}, {}
with threadpool_limits(limits=1):
    for mode in modes:
        fit_x, val_x, tr = transform(train, september, mode)
        scores[mode] = svm_scores(fit_x, y[train], val_x, chosen[mode])
        pick, hits, recall, ap = metrics(labels, scores[mode])
        top_sets[mode] = set(pick.tolist())
        print(f"{mode}: {fit_x.shape[1]} inputs | {hits}/{labels.sum()} failures in top {len(pick)} | recall {recall:.2%} | AP {ap:.5f}")
        if mode == "PCA-4":
            print(f"Train variance captured by 4 components: {tr[-1].explained_variance_ratio_.sum():.2%}")
print(f"Positive fraction = {labels.mean():.5f}; expected hits under random ranking = {labels.mean()*len(pick):.2f}")
common = top_sets[modes[0]] & top_sets[modes[1]]
print(f"Overlap among the top lists: {len(common)}; failures in the overlap: {labels[sorted(common)].sum()}")

All sensors: 452 inputs | 2/17 failures in top 59 | recall 11.76% | AP 0.03052
PCA-4: 4 inputs | 3/17 failures in top 59 | recall 17.65% | AP 0.04601
Train variance captured by 4 components: 17.09%
Positive fraction = 0.02881; expected hits under random ranking = 1.70
Overlap among the top lists: 14; failures in the overlap: 0


## 4큐비트 양자 커널의 개발 단계 비교

9월의 고전 모델 결과 확인 후, 양자 모델 결과를 보기 전에 다음 회로와 탐색 범위를 지정했다. 위 PCA-4 모델과 같은 훈련 행, 같은 전처리, 같은 4개 주성분 점수를 사용한다. 모델당 3 × 3 후보와 동일한 8월 내부 검증 구간을 사용한다. 9월 비교는 탐색적이다.

각 PCA 점수 벡터를 학습 구간에서 계산한 공통 RMS로 나누고 전체에 배율 0.5, 1, 2 중 하나를 곱한다. 생성 벡터를 u라 할 때 |0000⟩에서 시작해 다음 층을 두 번 실행한다. 모든 큐비트에 H 게이트, 각 큐비트 j에 RZ(uⱼ), 인접 큐비트 j, j+1에 RZZ(uⱼuⱼ₊₁)를 적용한다. 두 상태의 내적 절댓값 제곱이 SVM 커널이다. 아래 코드는 잡음 없는 16차원 상태벡터를 CPU로 계산한다. 물리 장치 실행 시간이나 측정 잡음은 이 실험 범위 밖이다.

In [4]:
from scipy.linalg import hadamard

n_qubits = 4
side = 2 ** n_qubits
H = hadamard(side).astype(float) / np.sqrt(side)
bits = (np.arange(side)[:, None] >> np.arange(n_qubits-1, -1, -1)) & 1
z_sign = 1 - 2*bits
zz_sign = z_sign[:, :-1] * z_sign[:, 1:]

def statevectors(inputs):
    phase = inputs @ z_sign.T + (inputs[:, :-1] * inputs[:, 1:]) @ zz_sign.T
    diagonal = np.exp(-0.5j * phase)
    states = (diagonal / np.sqrt(side)) @ H * diagonal
    assert np.allclose(np.linalg.norm(states, axis=1), 1)
    return states

def fidelity(left, right):
    return np.abs(left @ right.conj().T) ** 2

def gram_pair(fit_x, eval_x, scale):
    rms = np.sqrt(np.mean(fit_x ** 2))
    sf = statevectors(scale * fit_x / rms)
    sv = statevectors(scale * eval_x / rms)
    kfit = fidelity(sf, sf)
    assert np.allclose(kfit.diagonal(), 1) and np.allclose(kfit, kfit.T)
    return kfit, fidelity(sv, sf)

def fit_score_kernel(kfit, fit_y, keval, c):
    svm = SVC(C=c, kernel="precomputed", class_weight="balanced")
    svm.fit(kfit, fit_y)
    return svm.decision_function(keval)

# Check the vectorized calculation using explicit single-qubit gates and two-qubit phase gates.
probe = np.array([0.2, -0.7, 0.4, 1.1])
single_z = np.array([1.0, -1.0])
def kron_all(mats):
    acc = np.array([[1.0]])
    for mat in mats:
        acc = np.kron(acc, mat)
    return acc
explicit_h = kron_all([np.array([[1, 1], [1, -1]]) / np.sqrt(2)] * n_qubits)
state = np.eye(side, dtype=complex)[:, 0]
for _ in range(2):
    state = explicit_h @ state
    state = kron_all([np.diag(np.exp(-0.5j * angle * single_z)) for angle in probe]) @ state
    for j in range(n_qubits-1):
        pair_gate = np.diag(np.exp(-0.5j * probe[j] * probe[j+1] * np.kron(single_z, single_z)))
        state = kron_all([np.eye(2)]*j + [pair_gate] + [np.eye(2)]*(n_qubits-2-j)) @ state
assert np.allclose(state, statevectors(probe[None, :])[0], atol=1e-12)

scales = (0.5, 1.0, 2.0)
costs = (0.1, 1.0, 10.0)
ap_q = {(c, s): [] for c in costs for s in scales}
with threadpool_limits(limits=1):
    for start, end in fold_dates:
        fit = dates < np.datetime64(start)
        val = (dates >= np.datetime64(start)) & (dates < np.datetime64(end))
        fit_x, val_x, _ = transform(fit, val, "PCA-4")
        for s in scales:
            kfit, keval = gram_pair(fit_x, val_x, s)
            for c in costs:
                pred = fit_score_kernel(kfit, y[fit], keval, c)
                ap_q[c, s].append(average_precision_score(y[val], pred))
    best_q = max(ap_q, key=lambda p: np.mean(ap_q[p]))
    fit_x, val_x, _ = transform(train, september, "PCA-4")
    kfit, keval = gram_pair(fit_x, val_x, best_q[1])
    scores["Quantum-4"] = fit_score_kernel(kfit, y[train], keval, best_q[0])
print("Quantum-4 C / phase scale:", best_q, "| fold AP:", np.round(ap_q[best_q], 4))
q_top, hits_q, recall_q, ap_final_q = metrics(labels, scores["Quantum-4"])
top_sets["Quantum-4"] = set(q_top.tolist())
print(f"Quantum-4: {hits_q}/{labels.sum()} failures in top {len(q_top)} | recall {recall_q:.2%} | AP {ap_final_q:.5f}")
for mode in modes:
    common = top_sets[mode] & top_sets["Quantum-4"]
    print(mode, "and Quantum-4: overlap in top list =", len(common), "| failures in overlap =", int(labels[sorted(common)].sum()))

Quantum-4 C / phase scale: (1.0, 0.5) | fold AP: [0.406  0.1648 0.0898]
Quantum-4: 4/17 failures in top 59 | recall 23.53% | AP 0.05963
All sensors and Quantum-4: overlap in top list = 10 | failures in overlap = 0
PCA-4 and Quantum-4: overlap in top list = 45 | failures in overlap = 3


## 같은 날짜를 함께 재표집한 불확실성

9월에 관측된 날짜를 복원추출한 뒤 세 모델에 같은 행들을 적용한다. 매 반복에서 상위 10%를 다시 정해 압축 전후 및 두 커널 간 재현율·AP 차이를 계산한다. 백분위 95% 구간은 9월과 고정된 학습 결과에 조건부인 탐색적 지표다. 시점 간 의존, 다른 공정·월, 모델 재학습 변동까지 설명하지는 않는다.

In [5]:
rng = np.random.default_rng(20261123)
val_days = dates[september]
day_groups = [np.flatnonzero(val_days == day) for day in np.unique(val_days)]
diffs = []
for _ in range(2000):
    sampled = rng.integers(0, len(day_groups), size=len(day_groups))
    idx = np.sort(np.concatenate([day_groups[i] for i in sampled]), kind="stable")
    if not labels[idx].any():
        continue
    all_m, pca_m, quantum_m = [metrics(labels[idx], scores[mode][idx]) for mode in (*modes, 'Quantum-4')]
    diffs.append([pca_m[2] - all_m[2], pca_m[3] - all_m[3], quantum_m[2] - pca_m[2], quantum_m[3] - pca_m[3]])
interval = np.quantile(diffs, [0.025, 0.975], axis=0)
print("Bootstrap draws used:", len(diffs), "| observed days:", len(day_groups))
print("PCA-4 minus all sensors, recall (percentage points), 95%:", np.round(interval[:, 0] * 100, 2))
print("PCA-4 minus all sensors, AP, 95%:", np.round(interval[:, 1], 5))
print("Quantum-4 minus PCA-4, recall (percentage points), 95%:", np.round(interval[:, 2] * 100, 2))
print("Quantum-4 minus PCA-4, AP, 95%:", np.round(interval[:, 3], 5))

Bootstrap draws used: 2000 | observed days: 29
PCA-4 minus all sensors, recall (percentage points), 95%: [-18.2  30. ]
PCA-4 minus all sensors, AP, 95%: [-0.01315  0.05483]
Quantum-4 minus PCA-4, recall (percentage points), 95%: [ 0.   26.67]
Quantum-4 minus PCA-4, AP, 95%: [-0.01299  0.11848]


## 9월 파일럿 해석

상위 59건 중 포착한 불량은 전체 센서 RBF-SVM 2건, PCA-4 RBF-SVM 3건, 4큐비트 양자 커널 SVM 4건이다. 같은 입력의 두 커널이 공통으로 고른 건은 45건이며 이 안에 불량 3건이 있다. 양자 커널이 추가로 찾은 불량은 1건이었다.

두 커널의 AP 차이는 약 +0.0136이며, 날짜 단위 부트스트랩의 95% 구간은 약 −0.0130~+0.1185다. 9월에 관찰한 차이의 방향과 크기는 추가 검증 대상이다. 10월 평가 전에 회로, 지표, 검사 비율, 튜닝 및 최종 재학습 시점을 확정하는 것이 다음 단계다. 이 자료는 단일 공정 데이터의 한 달을 평가한 탐색 실험이며, 실제 장비 실험이나 실시간 경보 검증으로 해석할 수 있는 범위는 다르다.